# Modélisation et Explicabilité (v2)

Ce notebook entraîne un modèle de prédiction de survie et utilise SHAP/LIME pour l'interprétation.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import joblib
import importlib

# Robust path setup
project_root = Path(os.getcwd())
if project_root.name == 'notebooks':
    project_root = project_root.parent

src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.append(src_path)

print(f"Racine du projet : {project_root}")

# Import common components
from xai_clinical.data.pancan_loader import PANCANLoader, BRCALoader, PANCANBRCAFusion
from xai_clinical.models.classifiers import ClassifierFactory
from xai_clinical.explainability.shap_explainer import SHAPExplainer
from xai_clinical.explainability.lime_explainer import LIMEExplainer


## 1. Chargement des Données Préparées

In [ ]:
processed_path = project_root / "data" / "processed" / "processed_data_v2.pkl"
data = joblib.load(processed_path)

X_train, y_train = data['X_train'], data['y_train']
X_val, y_val = data['X_val'], data['y_val']
X_test, y_test = data['X_test'], data['y_test']
feature_names = data['feature_names']

print(f"Données chargées : Train={X_train.shape}, Test={X_test.shape}")

## 2. Entraînement du Modèle

On utilise le modèle LightGBM par défaut.

In [ ]:
factory = ClassifierFactory()
model = factory.create_classifier("lightgbm")
model.fit(X_train, y_train)

from sklearn.metrics import roc_auc_score, classification_report
y_pred_proba = model.predict_proba(X_test)[:, 1]
print(f"AUC Test : {roc_auc_score(y_test, y_pred_proba):.4f}")

## 3. Explicabilité avec SHAP

Grâce à notre correctif `_get_1d_shap_values`, les plots fonctionnent directement.

In [ ]:
shap_exp = SHAPExplainer(model, X_train, feature_names=feature_names)
shap_exp.fit()

print("Génération du Summary Plot...")
shap_exp.plot_summary(X_test)
plt.show()

In [ ]:
print("Explication locale (Waterfall plot) pour le premier échantillon du test...")
shap_exp.plot_waterfall(X_test.iloc[0], instance_index=0)
plt.show()


## 4. Comparaison SHAP vs LIME

Vérification de la cohérence entre les deux méthodes.

In [ ]:
lime_exp = LIMEExplainer(model, X_train, feature_names=feature_names)
lime_exp.compare_with_shap(X_test.iloc[0], shap_exp)
plt.show()
